# 3D Scan Pipeline - Part 2: Training (Nerfstudio Version)

**Objective:** Train a Gaussian Splatting model using Nerfstudio (Splatfacto) and Export to .splat.
**Input:** Use the output from Part 1 (`3d_scan_data_part1.zip`) as a **Kaggle Dataset**.

**How to use on Kaggle:**
1. In Part 1, download `3d_scan_data_part1.zip`.
2. Create a specific Dataset in Kaggle with this file.
3. Add your new Dataset to this notebook (Part 2).

**Environment:** **GPU REQUIRED (T4 or better)**.

In [ ]:
import os
import sys

print("⏳ Setting up Environment (Part 2: Nerfstudio)...")

# 1. Clone Repo (Only if local project not found)
if not os.path.exists("3DSCAN"):
    !git clone https://github.com/PRIDA-TAKON/3DSCAN.git
    if os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")
else:
    print("📂 Project folder found. Using local version.")
    if os.path.basename(os.getcwd()) != "3DSCAN" and os.path.exists("3DSCAN"):
        os.chdir("3DSCAN")

# 2. Install Nerfstudio Dependencies
print("⏳ Installing Nerfstudio & Dependencies...")
!pip install --upgrade pip
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!pip install ninja git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch
!pip install nerfstudio
!pip install plyfile

print("✅ Nerfstudio Setup Complete.")

In [ ]:
print("=== Import Data from Kaggle Dataset ===")
import glob
import zipfile
import shutil

# Reset working_data to avoid conflicts
if os.path.exists("working_data/3d_scan"):
    shutil.rmtree("working_data/3d_scan", ignore_errors=True)
os.makedirs("working_data/3d_scan", exist_ok=True)

# Look for the dataset zip file in /kaggle/input (standard dataset path)
search_paths = ["/kaggle/input", "input", "."]
zip_path = None

for path in search_paths:
    candidates = glob.glob(f"{path}/**/*.zip", recursive=True)
    for c in candidates:
        if "3d_scan_data_part1" in c or "3d_scan_output" in c:
            zip_path = c
            break
    if zip_path: break

folder_candidates = glob.glob("/kaggle/input/**/sparse/0", recursive=True)
if zip_path:
    print(f"📦 Found data zip: {zip_path}")
    print("⏳ Extracting to working_data...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("✅ Extraction Complete.")
elif folder_candidates:
    print("📂 Found unzipped dataset structure. Copying...")
    sparse_0_path = folder_candidates[0]
    current_path = os.path.dirname(sparse_0_path) # .../sparse
    root_path = None
    for _ in range(4):
        parent = os.path.dirname(current_path)
        if os.path.exists(os.path.join(parent, "images")) or os.path.exists(os.path.join(parent, "transforms.json")):
            root_path = parent
            break
        current_path = parent
    
    if root_path:
        print(f"   Detected Source Root: {root_path}")
        shutil.copytree(root_path, "working_data/3d_scan", dirs_exist_ok=True)
        print("✅ Data copied successfully.")
else:
    print("❌ No data found! Please add the '3d_scan_data_part1' dataset.")

In [ ]:
print("=== STEP 3: Train Nerfstudio (Splatfacto) ===")
if not os.path.exists("working_data/3d_scan"):
    print("❌ working_data/3d_scan not found.")
else:
    !python scripts/step3_train_splatting.py --project_path "working_data/3d_scan" --output_path "outputs/3d_scan/nerfstudio" --iterations 30000

In [ ]:
print("=== STEP 4: Export to SPLAT ===")
!python scripts/step4_export.py --input_config "outputs/3d_scan/nerfstudio" --output_splat "outputs/3d_scan/nerfstudio/model.splat"

In [ ]:
print("=== Compress Final Model for Download ===")
output_model_zip = "3d_nerfstudio_model.zip"
if os.path.exists("outputs/3d_scan/nerfstudio"):
    !zip -r {output_model_zip} outputs/3d_scan/nerfstudio
    from IPython.display import FileLink
    display(FileLink(output_model_zip))
else:
    print("❌ No output model found.")